# Лабораторная работа №1


Kaggle: https://www.kaggle.com/datasets/hubertsidorowicz/steam-games-dataset-daily-updates

В этом блокноте будем проводить очистку датасета и последующее сохранение в Iceberg.


## 1) Инициализация Spark


In [1]:
from src.spark_session import create_spark

spark = create_spark("Lab_1_Data_Cleaning")
spark

Picked up JAVA_TOOL_OPTIONS: -Djava.net.preferIPv4Stack=true
Picked up JAVA_TOOL_OPTIONS: -Djava.net.preferIPv4Stack=true


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/ivy/cache
The jars for the packages stored in: /tmp/ivy/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c2a2bc3-4507-4193-bdd7-9e7c77163fb3;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 in central
:: resolution report :: resolve 207ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-p

## 2) Пути и имя таблицы


In [2]:
INPUT_PATH = "/app/data/raw/steam_games.csv"
TABLE_NAME = "local.lab1.steam_games"

# None — использовать весь датасет.
# Для тестового запуска можно указать, например, 100_000.
ROW_LIMIT = None

print("CSV:", INPUT_PATH)
print("Iceberg:", TABLE_NAME)
print("Limit:", ROW_LIMIT)


CSV: /app/data/raw/steam_games.csv
Iceberg: local.lab1.steam_games
Limit: None


## 3) Чтение исходного CSV

В исходных текстовых полях могут встречаться переносы строк, поэтому CSV читается с `multiLine=true`.


In [3]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(INPUT_PATH)
)

if ROW_LIMIT is not None:
    df_raw = df_raw.limit(ROW_LIMIT)

# В исходном CSV у первого столбца с Steam App ID пустой заголовок.
first_column = df_raw.columns[0]
if first_column != "app_id":
    df_raw = df_raw.withColumnRenamed(first_column, "app_id")

print("Количество столбцов:", len(df_raw.columns))
print("Количество строк:", df_raw.count())
print("Количество партиций:", df_raw.rdd.getNumPartitions())
df_raw.printSchema()


Количество столбцов: 42


Количество строк: 140243
Количество партиций: 1
root
 |-- app_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- price_status: string (nullable = true)
 |-- estimated_owners: string (nullable = true)
 |-- developers: string (nullable = true)
 |-- publishers: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- positive: string (nullable = true)
 |-- negative: string (nullable = true)
 |-- recommendations: string (nullable = true)
 |-- peak_ccu: string (nullable = true)
 |-- metacritic_score: string (nullable = true)
 |-- user_score: string (nullable = true)
 |-- average_playtime_forever: string (nullable = true)
 |-- median_playtime_forever: string (nullable = true)
 |-- average_playtime_2weeks: string (nullable = true)
 |-- median_playtime_2weeks: string (nullable = true)
 |-- short_description:

## 4) Проверка корректности чтения


In [4]:
from pyspark.sql.functions import col, sum as spark_sum, when

df_raw.select(
    "app_id",
    "name",
    "genres",
    "categories",
    "tags",
    "positive",
    "negative",
    "price",
).show(10, truncate=False)

# Быстрая проверка, что записи CSV не разъехались из-за переносов строк.
df_raw.select(
    spark_sum(when(col("app_id").isNull(), 1).otherwise(0)).alias("app_id_null"),
    spark_sum(when(col("name").isNull(), 1).otherwise(0)).alias("name_null"),
    spark_sum(when(col("release_date").isNull(), 1).otherwise(0)).alias("release_date_null"),
).show()


+------+--------------------+-------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------+-----+
|app_id|name                |genres                                                       |categories                                                                                                                                                           |tags                                                                                                                                                                                                                     

+-----------+---------+-----------------+
|app_id_null|name_null|release_date_null|
+-----------+---------+-----------------+
|          0|       16|               83|
+-----------+---------+-----------------+



## 5) Выгрузка 1000 строк для ручного просмотра (не проводить)

In [13]:
from pathlib import Path

SAMPLE_OUTPUT_PATH = "/app/output/results/steam_sample_1000.csv"

sample_columns = [
    "app_id",
    "name",
    "release_date",
    "price",
    #"price_status",
    "estimated_owners", # разбить на 3 колонки (минимум, максимум, среднее)
    #"developers",
    #"publishers",
    "genres", # категории игр
    "categories",
    "tags",
    "positive", # позтивные и негативные комменты под играми (мб ввести отношение их)
    "negative",
    "recommendations",
    "peak_ccu",
    "metacritic_score",
    #"user_score",
    "average_playtime_forever",
    "median_playtime_forever",
    #"average_playtime_2weeks",
    #"median_playtime_2weeks",
    "achievements",
    "dlc_count",
    "supported_languages", # заменить на количество языков
    "full_audio_languages", # заменить на количество языков
    "windows",
    "mac",
    "linux",
    #"steam_store_available",
    #"steam_spy_available",
]

df_sample = df_raw.select(*sample_columns)#.limit(1500)

Path(SAMPLE_OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
df_sample.toPandas().to_csv(
    SAMPLE_OUTPUT_PATH,
    index=False,
    encoding="utf-8",
)

print("CSV сохранён:", SAMPLE_OUTPUT_PATH)
print("Строк:", df_sample.count())
print("Столбцов:", len(df_sample.columns))


CSV сохранён: /app/output/results/steam_sample_1000.csv


Строк: 140243
Столбцов: 22


## 6) Выбор признаков

In [5]:
SELECTED_COLUMNS = [
    "app_id",
    "name",
    "release_date",
    "price",
    "estimated_owners", # разбить на 3 колонки (минимум, максимум, среднее)
    "genres", # категории игр
    "categories",
    "tags",
    "positive", # позтивные и негативные комменты под играми (мб ввести отношение их)
    "negative",
    "recommendations",
    "peak_ccu",
    "metacritic_score",
    "average_playtime_forever",
    "median_playtime_forever",
    "achievements",
    "dlc_count",
    "supported_languages", # заменить на количество языков
    "full_audio_languages", # заменить на количество языков
    "windows",
    "mac",
    "linux",
]

df = df_raw.select(*SELECTED_COLUMNS)

df.show(10, truncate=False)

+------+--------------------+------------+-----+----------------+-------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------+---------------+--------+----------------+------------------------+-----------------------+------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 7) Приведение типов и преобразование признаков


In [6]:
from pyspark.sql.functions import (
    col,
    from_json,
    lit,
    regexp_extract,
    size,
    to_date,
    when,
)

from pyspark.sql.types import (
    ArrayType,
    StringType,
)


array_schema = ArrayType(StringType())


df = (
    df

    # -------------------------------------------------
    # Основные признаки
    # -------------------------------------------------

    .withColumn(
        "app_id",
        col("app_id").cast("long"),
    )

    .withColumn(
        "release_date",
        to_date(
            col("release_date"),
            "yyyy-MM-dd",
        ),
    )

    .withColumn(
        "price",
        col("price").cast("double"),
    )

    # -------------------------------------------------
    # estimated_owners
    # "50000 - 100000" -> min / max / avg
    # -------------------------------------------------

    .withColumn(
        "estimated_owners_min",
        regexp_extract(
            col("estimated_owners"),
            r"^\s*(\d+)",
            1,
        ).cast("long"),
    )

    .withColumn(
        "estimated_owners_max",
        regexp_extract(
            col("estimated_owners"),
            r"(\d+)\s*$",
            1,
        ).cast("long"),
    )

    .withColumn(
        "estimated_owners_avg",
        (
            col("estimated_owners_min")
            + col("estimated_owners_max")
        ) / 2,
    )

    # -------------------------------------------------
    # Множественные категориальные признаки
    # string -> array<string>
    # -------------------------------------------------

    .withColumn(
        "genres",
        from_json(
            col("genres"),
            array_schema,
        ),
    )

    .withColumn(
        "categories",
        from_json(
            col("categories"),
            array_schema,
        ),
    )

    .withColumn(
        "tags",
        from_json(
            col("tags"),
            array_schema,
        ),
    )

    # -------------------------------------------------
    # Отзывы и активность
    # -------------------------------------------------

    .withColumn(
        "positive",
        col("positive").cast("long"),
    )

    .withColumn(
        "negative",
        col("negative").cast("long"),
    )

    .withColumn(
        "recommendations",
        col("recommendations")
        .cast("double")
        .cast("long"),
    )

    .withColumn(
        "peak_ccu",
        col("peak_ccu").cast("long"),
    )

    .withColumn(
        "metacritic_score",
        col("metacritic_score")
        .cast("double")
        .cast("integer"),
    )

    # -------------------------------------------------
    # Время игры
    # -------------------------------------------------

    .withColumn(
        "average_playtime_forever",
        col("average_playtime_forever").cast("long"),
    )

    .withColumn(
        "median_playtime_forever",
        col("median_playtime_forever").cast("long"),
    )

    # -------------------------------------------------
    # Достижения и DLC
    # -------------------------------------------------

    .withColumn(
        "achievements",
        col("achievements")
        .cast("double")
        .cast("integer"),
    )

    .withColumn(
        "dlc_count",
        col("dlc_count").cast("integer"),
    )

    # -------------------------------------------------
    # Языки
    # -------------------------------------------------

    .withColumn(
        "_supported_languages",
        from_json(
            col("supported_languages"),
            array_schema,
        ),
    )

    .withColumn(
        "_full_audio_languages",
        from_json(
            col("full_audio_languages"),
            array_schema,
        ),
    )

    # [] у supported_languages считаем отсутствием данных
    .withColumn(
        "supported_languages_count",
        when(
            col("_supported_languages").isNull()
            | (size(col("_supported_languages")) == 0),
            lit(None).cast("integer"),
        ).otherwise(
            size(col("_supported_languages"))
        ),
    )

    # [] у full_audio_languages = озвучки нет -> 0
    .withColumn(
        "full_audio_languages_count",
        when(
            col("_full_audio_languages").isNull(),
            lit(None).cast("integer"),
        ).otherwise(
            size(col("_full_audio_languages"))
        ),
    )

    # -------------------------------------------------
    # Поддержка ОС
    # -------------------------------------------------

    .withColumn(
        "windows",
        col("windows").cast("boolean"),
    )

    .withColumn(
        "mac",
        col("mac").cast("boolean"),
    )

    .withColumn(
        "linux",
        col("linux").cast("boolean"),
    )

    # -------------------------------------------------
    # Удаляем признаки, которые уже преобразовали
    # -------------------------------------------------

    .drop(
        "estimated_owners",
        "supported_languages",
        "full_audio_languages",
        "_supported_languages",
        "_full_audio_languages",
    )
)


df.printSchema()

root
 |-- app_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- recommendations: long (nullable = true)
 |-- peak_ccu: long (nullable = true)
 |-- metacritic_score: integer (nullable = true)
 |-- average_playtime_forever: long (nullable = true)
 |-- median_playtime_forever: long (nullable = true)
 |-- achievements: integer (nullable = true)
 |-- dlc_count: integer (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- estimated_owners_min: long (nullable = true)
 |-- estimated_owners

## 8) Проверка результата


In [7]:
df.show(10, truncate=False)

+------+--------------------+------------+-----+-------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------+---------------+--------+----------------+------------------------+-----------------------+------------+---------+-------+-----+-----+--------------------+--------------------+--------------------+-------------------------+--------------------------+
|app_id|name                |release_date|price|genres                                           |categories                                                                                                                                       |tags                                   

## 9) Сохранение в Iceberg


In [8]:
spark.sql(
    "CREATE NAMESPACE IF NOT EXISTS local.lab1"
)

(
    df.writeTo(TABLE_NAME)
    .using("iceberg")
    .createOrReplace()
)

print("Таблица успешно сохранена:", TABLE_NAME)

Таблица успешно сохранена: local.lab1.steam_games


## 10) Проверка сохранённой таблицы


In [9]:
df_saved = spark.table(TABLE_NAME)

print("Количество строк:", f"{df_saved.count():,}")
print("Количество столбцов:", len(df_saved.columns))

df_saved.printSchema()
df_saved.show(10, truncate=False)

Количество строк: 140,243
Количество столбцов: 24
root
 |-- app_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- recommendations: long (nullable = true)
 |-- peak_ccu: long (nullable = true)
 |-- metacritic_score: integer (nullable = true)
 |-- average_playtime_forever: long (nullable = true)
 |-- median_playtime_forever: long (nullable = true)
 |-- achievements: integer (nullable = true)
 |-- dlc_count: integer (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- estimated_owners

## 11) Проверка каталога Iceberg


In [10]:
spark.sql(
    "SHOW TABLES IN local.lab1"
).show(truncate=False)


+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|lab1     |steam_games|false      |
|lab1     |test_cars  |false      |
+---------+-----------+-----------+



## 12) Просмотр данных в таблице Iceberg


In [11]:
spark.table("local.lab1.steam_games").show(10, truncate=False)

+------+--------------------+------------+-----+-------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------+---------------+--------+----------------+------------------------+-----------------------+------------+---------+-------+-----+-----+--------------------+--------------------+--------------------+-------------------------+--------------------------+
|app_id|name                |release_date|price|genres                                           |categories                                                                                                                                       |tags                                   